# Expert validation 3: ranker vs raters

This notebook compares:
- each rater vs the ranker, and
- raters vs each other

for the papers selected by notebook 01.

## Ratings

The repository contains no ratings: supply your own. Put one CSV file per rater in
`outputs/expert_validation/ratings/`, with these columns:

- `paper_id`: the paper key from notebook 02's PDF names (`<seed and timestamp>__<paper id>`), or
  the full paper UID (`<snapshot dir>::<paper id>`);
- `novelty_rank`, `significance_rank`, `soundness_rank`, `overall_rank`: the rater's rank of the
  paper among the selected papers on each criterion, 1 = best;
- `notes` (optional).

Raters are named Rater A, Rater B, ... in file-name order.

## How to use
1. Add the ratings files.
2. Run all cells.
3. Inspect:
   - `summary_expert_vs_ranker`
   - `summary_expert_vs_expert`
   - `detail_by_expert`

In [ ]:
import itertools
import pathlib
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

REPO_ROOT = pathlib.Path("../..").resolve()
OUT_DIR = REPO_ROOT / "outputs/expert_validation"

# Selected papers with the ranker's win rates, exported by notebook 01.
RANKER_SELECTION_PATH = OUT_DIR / "selection.json"

# One ratings CSV per rater (see above); raters are named in file-name order.
RATINGS_DIR = OUT_DIR / "ratings"
EXPERT_CSV_PATHS = sorted(RATINGS_DIR.glob("*.csv"))
RATER_NAMES = {p: f"Rater {chr(ord('A') + i)}" for i, p in enumerate(EXPERT_CSV_PATHS)}

if not EXPERT_CSV_PATHS:
    raise FileNotFoundError(
        f"No ratings CSV files in {RATINGS_DIR}: add one per rater (see above)."
    )

assert RANKER_SELECTION_PATH.exists(), f"Missing {RANKER_SELECTION_PATH}: run notebook 01 first"

print(f"Repo root:        {REPO_ROOT}")
print(f"Ranker selection: {RANKER_SELECTION_PATH.relative_to(REPO_ROOT)}")
print("Raters:")
for p, name in RATER_NAMES.items():
    print(f"  {name}: {p.name}")

In [ ]:
def extract_seed_timestamp(snapshot_dir_or_uid: str) -> str:
    text = str(snapshot_dir_or_uid)
    m = re.search(r"(seed_\d+_\d{6}_\d{4})", text)
    if m:
        return m.group(1)
    m = re.search(r"(seed_\d+)", text)
    if m:
        return m.group(1)
    return "unknown_seed"


def to_short_paper_id_from_ranker(snapshot_dir: str, paper_id: str) -> str:
    return f"{extract_seed_timestamp(snapshot_dir)}__{paper_id}"


def normalize_expert_paper_id(raw_id: str) -> str:
    """Accepts either short id or full UID and returns short id."""
    raw = str(raw_id)

    # Already short form: seed_...__paper_...
    if re.match(r"^seed_\d+(?:_\d{6}_\d{4})?__paper_", raw):
        return raw

    # Full UID form: snapshots/...seed...::paper_...
    if "::" in raw:
        left, right = raw.split("::", 1)
        return f"{extract_seed_timestamp(left)}__{right}"

    # Fallback: return as-is.
    return raw


required_expert_cols = [
    "paper_id",
    "novelty_rank",
    "significance_rank",
    "soundness_rank",
    "overall_rank",
]

ranker_df = pd.read_json(RANKER_SELECTION_PATH).copy()
required_ranker_cols = [
    "paper_id",
    "source_snapshot_dir",
    "wr_novelty",
    "wr_significance",
    "wr_soundness",
    "wr_default",
]
missing_ranker = [c for c in required_ranker_cols if c not in ranker_df.columns]
if missing_ranker:
    raise ValueError(f"Ranker selection JSON missing required columns: {missing_ranker}")

ranker_df["paper_id_short"] = ranker_df.apply(
    lambda r: to_short_paper_id_from_ranker(r["source_snapshot_dir"], r["paper_id"]),
    axis=1,
)
if ranker_df["paper_id_short"].duplicated().any():
    dups = ranker_df.loc[
        ranker_df["paper_id_short"].duplicated(keep=False), "paper_id_short"
    ].tolist()
    raise ValueError(f"Duplicate short paper IDs in ranker data: {dups[:10]}")

print(f"Ranker rows: {len(ranker_df)}")
print("Ranker columns sample:", list(ranker_df.columns)[:25])

In [ ]:
def strict_rank_from_score(
    df: pd.DataFrame, score_col: str, id_col: str = "paper_id_short"
) -> pd.Series:
    """Create deterministic strict rank: sort by score desc, then id asc."""
    ordered = (
        df[[id_col, score_col]]
        .sort_values([score_col, id_col], ascending=[False, True])
        .reset_index(drop=True)
    )
    return pd.Series(
        np.arange(1, len(ordered) + 1), index=ordered[id_col], name=f"rank_{score_col}"
    )


# Ranker strict ranks on the whole selected pool.
score_cols = {
    "novelty": "wr_novelty",
    "significance": "wr_significance",
    "soundness": "wr_soundness",
    "overall_default": "wr_default",
}
if "wr_naive_overall" in ranker_df.columns:
    score_cols["overall_naive"] = "wr_naive_overall"

for label, score_col in score_cols.items():
    ranks = strict_rank_from_score(ranker_df, score_col, id_col="paper_id_short")
    ranker_df[f"ranker_rank__{label}"] = ranker_df["paper_id_short"].map(ranks)

experts: dict[str, pd.DataFrame] = {}
for p, name in RATER_NAMES.items():
    df = pd.read_csv(p).copy()
    missing = [c for c in required_expert_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Ratings file {p.name} missing required columns: {missing}")

    df["paper_id_short"] = df["paper_id"].map(normalize_expert_paper_id)
    if df["paper_id_short"].duplicated().any():
        dups = df.loc[df["paper_id_short"].duplicated(keep=False), "paper_id_short"].tolist()
        raise ValueError(f"Duplicate paper IDs in ratings file {p.name}: {dups[:10]}")

    for col in ["novelty_rank", "significance_rank", "soundness_rank", "overall_rank"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["expert_name"] = name
    df["expert_source_file"] = p.name
    experts[name] = df

print(f"Loaded {len(experts)} expert file(s): {list(experts.keys())}")
for name, df in experts.items():
    print(f"  {name}: {len(df)} rows")

In [ ]:
def pairwise_agreement_and_tau(rank_a: pd.Series, rank_b: pd.Series) -> dict[str, float]:
    """Compute pairwise agreement and tau-like score from two strict rank vectors."""
    a = rank_a.to_numpy()
    b = rank_b.to_numpy()

    concordant = 0
    discordant = 0
    n = len(a)

    for i in range(n):
        for j in range(i + 1, n):
            da = np.sign(a[i] - a[j])
            db = np.sign(b[i] - b[j])
            if da == db:
                concordant += 1
            else:
                discordant += 1

    total_pairs = concordant + discordant
    if total_pairs == 0:
        return {"pairwise_accuracy": np.nan, "kendall_tau": np.nan, "n_pairs": 0}

    return {
        "pairwise_accuracy": concordant / total_pairs,
        "kendall_tau": (concordant - discordant) / total_pairs,
        "n_pairs": total_pairs,
    }


def topk_overlap(df: pd.DataFrame, col_a: str, col_b: str, k: int) -> int:
    kk = min(k, len(df))
    if kk <= 0:
        return 0
    set_a = set(df.nsmallest(kk, col_a)["paper_id_short"])
    set_b = set(df.nsmallest(kk, col_b)["paper_id_short"])
    return len(set_a & set_b)


def compare_two_rankings(
    df: pd.DataFrame, col_a: str, col_b: str, id_col: str = "paper_id_short"
) -> dict[str, float]:
    sub = df[[id_col, col_a, col_b]].dropna().copy()
    if len(sub) == 0:
        return {
            "n": 0,
            "spearman_rho": np.nan,
            "mae_rank": np.nan,
            "rmse_rank": np.nan,
            "pairwise_accuracy": np.nan,
            "kendall_tau": np.nan,
            "top3_overlap": np.nan,
            "top5_overlap": np.nan,
        }

    pair = pairwise_agreement_and_tau(sub[col_a], sub[col_b])
    return {
        "n": len(sub),
        "spearman_rho": sub[col_a].corr(sub[col_b], method="spearman"),
        "mae_rank": (sub[col_a] - sub[col_b]).abs().mean(),
        "rmse_rank": np.sqrt(((sub[col_a] - sub[col_b]) ** 2).mean()),
        "pairwise_accuracy": pair["pairwise_accuracy"],
        "kendall_tau": pair["kendall_tau"],
        "top3_overlap": topk_overlap(sub, col_a, col_b, 3),
        "top5_overlap": topk_overlap(sub, col_a, col_b, 5),
    }

In [ ]:
# 1) Compare each expert vs ranker
comparison_specs = [
    ("novelty", "novelty_rank", "ranker_rank__novelty"),
    ("significance", "significance_rank", "ranker_rank__significance"),
    ("soundness", "soundness_rank", "ranker_rank__soundness"),
    ("overall_default", "overall_rank", "ranker_rank__overall_default"),
]
if "ranker_rank__overall_naive" in ranker_df.columns:
    comparison_specs.append(("overall_naive", "overall_rank", "ranker_rank__overall_naive"))

rows_vs_ranker: list[dict] = []
detail_by_expert: dict[str, pd.DataFrame] = {}
merged_by_expert: dict[str, pd.DataFrame] = {}

for expert_name, expert_df in experts.items():
    merged = expert_df.merge(
        ranker_df,
        on="paper_id_short",
        how="inner",
        suffixes=("_expert", "_ranker"),
    ).copy()

    merged_by_expert[expert_name] = merged

    missing_in_ranker = sorted(set(expert_df["paper_id_short"]) - set(merged["paper_id_short"]))
    missing_in_expert = sorted(set(ranker_df["paper_id_short"]) - set(merged["paper_id_short"]))

    print(f"\n[{expert_name}] matched papers: {len(merged)}")
    print(f"  missing in ranker: {len(missing_in_ranker)}")
    print(f"  missing in expert: {len(missing_in_expert)}")

    for dim_label, expert_col, ranker_col in comparison_specs:
        metrics = compare_two_rankings(merged, expert_col, ranker_col)
        rows_vs_ranker.append(
            {
                "expert": expert_name,
                "dimension": dim_label,
                **metrics,
            }
        )

    detail_cols = [
        "paper_id_short",
        "title",
        "overall_rank",
        "ranker_rank__overall_default",
        "wr_default",
    ] + (["notes"] if "notes" in merged.columns else [])
    detail = merged[detail_cols].copy()
    detail["rank_diff_default"] = detail["ranker_rank__overall_default"] - detail["overall_rank"]
    detail = detail.sort_values("overall_rank")
    detail_by_expert[expert_name] = detail

summary_expert_vs_ranker = (
    pd.DataFrame(rows_vs_ranker).sort_values(["expert", "dimension"]).reset_index(drop=True)
)
summary_expert_vs_ranker

In [ ]:
# 2) Compare experts vs each other (pairwise)
rows_expert_vs_expert: list[dict] = []

expert_pairs = list(itertools.combinations(sorted(experts.keys()), 2))
if not expert_pairs:
    print("Only one expert provided: expert-vs-expert table is empty.")

for a_name, b_name in expert_pairs:
    a = experts[a_name]
    b = experts[b_name]
    pair_df = a.merge(
        b,
        on="paper_id_short",
        how="inner",
        suffixes=(f"__{a_name}", f"__{b_name}"),
    ).copy()

    print(f"\n[{a_name} vs {b_name}] matched papers: {len(pair_df)}")

    dim_to_cols = {
        "novelty": (f"novelty_rank__{a_name}", f"novelty_rank__{b_name}"),
        "significance": (f"significance_rank__{a_name}", f"significance_rank__{b_name}"),
        "soundness": (f"soundness_rank__{a_name}", f"soundness_rank__{b_name}"),
        "overall": (f"overall_rank__{a_name}", f"overall_rank__{b_name}"),
    }

    for dim_label, (col_a, col_b) in dim_to_cols.items():
        metrics = compare_two_rankings(pair_df, col_a, col_b)
        rows_expert_vs_expert.append(
            {
                "expert_a": a_name,
                "expert_b": b_name,
                "dimension": dim_label,
                **metrics,
            }
        )

summary_expert_vs_expert = pd.DataFrame(rows_expert_vs_expert)
if len(summary_expert_vs_expert):
    summary_expert_vs_expert = summary_expert_vs_expert.sort_values(
        ["expert_a", "expert_b", "dimension"]
    ).reset_index(drop=True)

summary_expert_vs_expert

In [ ]:
# Publication-ready agreement plots: expert rankings vs ranker
# Primary figure: grouped Spearman bars. Alternatives: annotated heatmap and rank-difference diagnostic.
import matplotlib as mpl
import matplotlib.ticker as mticker
from matplotlib.colors import Normalize, TwoSlopeNorm

try:
    import scienceplots  # noqa: F401  # Registers the "science" Matplotlib style.

    PAPER_STYLE = ["science", "no-latex"]
except Exception:
    PAPER_STYLE = ["seaborn-v0_8-whitegrid"]

try:
    import seaborn as sns
except Exception:
    sns = None

DIMENSION_LABELS = {
    "novelty": "Novelty",
    "significance": "Significance",
    "soundness": "Soundness",
    "overall_default": "Overall\n(default)",
    "overall_naive": "Overall\n(naive)",
    "overall": "Overall",
}
PREFERRED_DIMENSION_ORDER = [
    "novelty",
    "significance",
    "soundness",
    "overall_default",
    "overall_naive",
    "overall",
]

PAPER_RC = {
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "font.size": 8.5,
    "axes.labelsize": 8.5,
    "axes.titlesize": 9.5,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7.5,
    "legend.title_fontsize": 8,
    "axes.linewidth": 0.7,
    "grid.linewidth": 0.5,
    "lines.linewidth": 1.2,
    "patch.linewidth": 0.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
}


def _ordered_dimensions(dimensions):
    present = list(pd.Series(dimensions).dropna().unique())
    ordered = [d for d in PREFERRED_DIMENSION_ORDER if d in present]
    ordered.extend(sorted(d for d in present if d not in ordered))
    return ordered


def _dimension_label(dimension):
    return DIMENSION_LABELS.get(dimension, str(dimension).replace("_", " ").title())


def _colorblind_palette(n):
    if sns is not None:
        return sns.color_palette("colorblind", n_colors=n)
    cmap = plt.get_cmap("tab10")
    return [cmap(i % cmap.N) for i in range(n)]


def _metric_limits(values):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if len(finite) == 0:
        return 0.0, 1.0, False
    lower = -1.0 if finite.min() < 0 else 0.0
    return lower, 1.0, lower < 0


def _matched_n_label(n_values):
    n_values = pd.Series(np.ravel(n_values)).dropna().astype(int).unique()
    if len(n_values) == 0:
        return "Matched papers: unavailable"
    if len(n_values) == 1:
        return f"Matched papers per comparison: n={n_values[0]}"
    return f"Matched papers per comparison: n={n_values.min()}-{n_values.max()}"


def _clean_title(value, max_chars=24):
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return ""
    return text if len(text) <= max_chars else text[: max_chars - 1].rstrip() + "..."


if len(summary_expert_vs_ranker):
    plot_df = summary_expert_vs_ranker.copy()
    dimension_order = _ordered_dimensions(plot_df["dimension"])
    expert_order = sorted(plot_df["expert"].dropna().unique())
    dimension_tick_labels = [_dimension_label(d) for d in dimension_order]

    spearman = plot_df.pivot_table(
        index="dimension", columns="expert", values="spearman_rho", aggfunc="mean"
    ).reindex(index=dimension_order, columns=expert_order)
    matched_n = plot_df.pivot_table(
        index="dimension", columns="expert", values="n", aggfunc="mean"
    ).reindex(index=dimension_order, columns=expert_order)

    # Keep the exact numbers visible for paper captions/tables.
    display(
        spearman.rename(index=_dimension_label)
        .rename_axis(index="Dimension", columns="Rater")
        .round(3)
    )

    lower, upper, has_negative = _metric_limits(spearman.to_numpy())
    palette = _colorblind_palette(len(expert_order))
    n_label = _matched_n_label(matched_n.to_numpy())

    with plt.style.context(PAPER_STYLE), mpl.rc_context(PAPER_RC):
        # Updated version of the current chart: grouped bars with direct numeric labels.
        bar_width = min(0.82 / max(len(expert_order), 1), 0.28)
        x = np.arange(len(dimension_order))
        fig_width = max(5.8, 0.95 * len(dimension_order) + 0.65 * len(expert_order))
        fig, ax = plt.subplots(figsize=(fig_width, 3.2))

        for i, expert in enumerate(expert_order):
            values = spearman[expert].to_numpy(dtype=float)
            offset = (i - (len(expert_order) - 1) / 2) * bar_width
            bars = ax.bar(
                x + offset,
                values,
                width=bar_width,
                label=expert,
                color=palette[i],
                edgecolor="0.2",
                linewidth=0.45,
            )
            if spearman.size <= 30:
                for bar, value in zip(bars, values, strict=True):
                    if not np.isfinite(value):
                        continue
                    pad = 0.025 if value >= 0 else -0.025
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        value + pad,
                        f"{value:.2f}",
                        ha="center",
                        va="bottom" if value >= 0 else "top",
                        fontsize=7.2,
                        clip_on=False,
                    )

        ax.axhline(0, color="0.2", linewidth=0.7)
        ax.set_ylim(lower - (0.08 if has_negative else 0.0), upper + 0.08)
        ax.set_yticks(np.linspace(lower, upper, 6 if not has_negative else 9))
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
        ax.set_ylabel("Spearman $\\rho$ vs. ranker")
        ax.set_xlabel("")
        ax.set_xticks(x)
        ax.set_xticklabels(dimension_tick_labels)
        ax.set_title("Expert-ranker rank correlation by scoring dimension", loc="left", pad=12)
        ax.text(0.0, 1.02, n_label, transform=ax.transAxes, fontsize=7.2, color="0.25")
        ax.grid(axis="y", color="0.86", linestyle="-", linewidth=0.55)
        ax.grid(axis="x", visible=False)
        ax.legend(
            title="Rater",
            frameon=False,
            loc="upper left",
            bbox_to_anchor=(1.01, 1.0),
            borderaxespad=0.0,
            handlelength=1.2,
        )
        if sns is not None:
            sns.despine(ax=ax)
        fig.tight_layout()
        plt.show()

        # Alternative 1: compact annotated heatmap. This is often the best main-paper panel
        # when space is tight or when more experts/dimensions are added.
        heat_values = spearman.to_numpy(dtype=float)
        fig_width = max(4.4, 0.72 * len(expert_order) + 2.4)
        fig_height = max(2.4, 0.36 * len(dimension_order) + 1.35)
        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        if has_negative:
            norm = TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)
            cmap = "coolwarm"
        else:
            norm = Normalize(vmin=0.0, vmax=1.0)
            cmap = "viridis"
        im = ax.imshow(heat_values, aspect="auto", cmap=cmap, norm=norm)

        for row_idx, dimension in enumerate(dimension_order):
            for col_idx, expert in enumerate(expert_order):
                value = spearman.loc[dimension, expert]
                if not np.isfinite(value):
                    ax.text(
                        col_idx, row_idx, "NA", ha="center", va="center", fontsize=7, color="0.45"
                    )
                    continue
                relative = (value - lower) / (upper - lower) if upper > lower else 1.0
                text_color = "white" if (not has_negative and relative < 0.42) else "black"
                ax.text(
                    col_idx,
                    row_idx,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7.5,
                    color=text_color,
                )

        ax.set_xticks(np.arange(len(expert_order)))
        ax.set_xticklabels(
            expert_order,
            rotation=35 if len(expert_order) > 3 else 0,
            ha="right" if len(expert_order) > 3 else "center",
        )
        ax.set_yticks(np.arange(len(dimension_order)))
        ax.set_yticklabels(dimension_tick_labels)
        ax.set_xlabel("Rater")
        ax.set_ylabel("Dimension")
        ax.set_title("Expert-ranker Spearman agreement matrix", loc="left", pad=10)
        ax.tick_params(length=0)
        ax.set_xticks(np.arange(-0.5, len(expert_order), 1), minor=False)
        ax.set_yticks(np.arange(-0.5, len(dimension_order), 1), minor=False)
        ax.grid(which="minor", color="white", linestyle="-", linewidth=1.0)
        ax.tick_params(which="minor", bottom=False, left=False)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
        cbar.set_label("Spearman $\\rho$")
        if sns is not None:
            sns.despine(ax=ax, left=True, bottom=True)
        fig.tight_layout()
        plt.show()

        # Alternative 2: rank-difference diagnostic for overall agreement.
        # Positive values mean the ranker placed the paper lower/worse than the expert did.
        rank_diff_dimension = (
            "overall_default" if "overall_default" in dimension_order else dimension_order[-1]
        )
        spec_lookup = {
            dim: (expert_col, ranker_col) for dim, expert_col, ranker_col in comparison_specs
        }
        rank_diff_rows = []
        if rank_diff_dimension in spec_lookup:
            expert_col, ranker_col = spec_lookup[rank_diff_dimension]
            for expert in expert_order:
                if expert not in merged_by_expert:
                    continue
                cols = ["paper_id_short", expert_col, ranker_col]
                if "title" in merged_by_expert[expert].columns:
                    cols.append("title")
                sub = merged_by_expert[expert][cols].dropna(subset=[expert_col, ranker_col]).copy()
                if not len(sub):
                    continue
                sub["expert"] = expert
                sub["expert_rank"] = sub[expert_col]
                sub["ranker_rank"] = sub[ranker_col]
                sub["mean_rank"] = (sub["expert_rank"] + sub["ranker_rank"]) / 2
                sub["ranker_minus_expert"] = sub["ranker_rank"] - sub["expert_rank"]
                sub["abs_diff"] = sub["ranker_minus_expert"].abs()
                if "title" not in sub.columns:
                    sub["title"] = sub["paper_id_short"]
                rank_diff_rows.append(
                    sub[
                        [
                            "expert",
                            "paper_id_short",
                            "title",
                            "expert_rank",
                            "ranker_rank",
                            "mean_rank",
                            "ranker_minus_expert",
                            "abs_diff",
                        ]
                    ]
                )

        if rank_diff_rows:
            rank_diff_df = pd.concat(rank_diff_rows, ignore_index=True)
            rank_diff_experts = [e for e in expert_order if e in set(rank_diff_df["expert"])]
            fig_width = max(5.8, 2.7 * len(rank_diff_experts))
            fig, axes = plt.subplots(
                1,
                len(rank_diff_experts),
                figsize=(fig_width, 3.0),
                sharex=True,
                sharey=True,
                squeeze=False,
            )
            axes = axes.ravel()
            max_mean_rank = np.ceil(rank_diff_df["mean_rank"].max())
            max_abs_diff = max(1.0, np.ceil(rank_diff_df["ranker_minus_expert"].abs().max()))

            for ax, expert, color in zip(axes, rank_diff_experts, palette, strict=False):
                sub = rank_diff_df.loc[rank_diff_df["expert"] == expert].copy()
                ax.scatter(
                    sub["mean_rank"],
                    sub["ranker_minus_expert"],
                    s=28,
                    color=color,
                    edgecolor="white",
                    linewidth=0.45,
                    alpha=0.88,
                )
                ax.axhline(0, color="0.2", linewidth=0.75)
                mean_bias = sub["ranker_minus_expert"].mean()
                ax.axhline(mean_bias, color=color, linestyle=":", linewidth=1.0)

                ax.text(
                    0.02,
                    0.96,
                    f"mean diff={mean_bias:.1f}",
                    transform=ax.transAxes,
                    ha="left",
                    va="top",
                    fontsize=6.8,
                    color="0.25",
                )

                ax.set_title(expert, pad=7)
                ax.set_xlabel("Mean of expert and ranker ranks")
                ax.set_xlim(0.5, max_mean_rank + 0.6)
                ax.set_ylim(-max_abs_diff - 0.6, max_abs_diff + 0.6)
                ax.grid(axis="y", color="0.86", linestyle="-", linewidth=0.55)
                ax.grid(axis="x", color="0.92", linestyle="-", linewidth=0.45)
                if sns is not None:
                    sns.despine(ax=ax)

            axes[0].set_ylabel("Ranker rank - expert rank\n(+ = ranker lower)")
            fig.suptitle(
                f"Rank-difference diagnostic: {_dimension_label(rank_diff_dimension).replace(chr(10), ' ')}",
                x=0.01,
                y=1.04,
                ha="left",
                fontsize=9.5,
            )
            fig.tight_layout()
            plt.show()

            top_disagreements = (
                rank_diff_df.sort_values(["expert", "abs_diff"], ascending=[True, False])
                .groupby("expert", as_index=False)
                .head(3)[
                    [
                        "expert",
                        "title",
                        "expert_rank",
                        "ranker_rank",
                        "ranker_minus_expert",
                        "abs_diff",
                    ]
                ]
                .copy()
            )
            top_disagreements["title"] = top_disagreements["title"].map(
                lambda x: _clean_title(x, max_chars=48)
            )
            display(
                top_disagreements.rename(
                    columns={
                        "expert": "Rater",
                        "title": "Largest-disagreement paper",
                        "expert_rank": "Rater rank",
                        "ranker_rank": "Ranker rank",
                        "ranker_minus_expert": "Ranker - rater",
                        "abs_diff": "|diff|",
                    }
                ).round({"Rater rank": 0, "Ranker rank": 0, "Ranker - rater": 0, "|diff|": 0})
            )
else:
    print("No expert-vs-ranker comparisons to plot.")

In [ ]:
# Combined agreement plot: expert-expert agreement alongside expert-ranker agreement
import matplotlib as mpl
from matplotlib.colors import Normalize, TwoSlopeNorm

try:
    import seaborn as sns
except Exception:
    sns = None

# PAPER_STYLE and PAPER_RC come from the previous cell.
TRIAD_RANKER_OVERALL_DIMENSION = "overall_default"
TRIAD_DIMENSION_LABELS = {
    "novelty": "Novelty",
    "significance": "Significance",
    "soundness": "Soundness",
    "overall_default": "Overall\n(default)",
    "overall_naive": "Overall\n(naive)",
    "overall": "Overall",
}


def _triad_dimension_label(dimension):
    return TRIAD_DIMENSION_LABELS.get(dimension, str(dimension).replace("_", " ").title())


if not len(summary_expert_vs_ranker):
    print("No expert-vs-ranker comparisons to plot.")
else:
    ranker_dimensions = set(summary_expert_vs_ranker["dimension"].dropna())
    if TRIAD_RANKER_OVERALL_DIMENSION not in ranker_dimensions:
        TRIAD_RANKER_OVERALL_DIMENSION = (
            "overall_naive" if "overall_naive" in ranker_dimensions else "overall"
        )

    dimension_order = [
        d
        for d in ["novelty", "significance", "soundness", TRIAD_RANKER_OVERALL_DIMENSION]
        if d in ranker_dimensions or d == TRIAD_RANKER_OVERALL_DIMENSION
    ]
    excluded_ranker_overall_dimensions = {"overall_default", "overall_naive"} - {
        TRIAD_RANKER_OVERALL_DIMENSION
    }

    agreement_rows = []
    for _, row in summary_expert_vs_ranker.iterrows():
        dimension = row["dimension"]
        if dimension in excluded_ranker_overall_dimensions or dimension not in dimension_order:
            continue
        agreement_rows.append(
            {
                "comparison": f"{row['expert']} vs Ranker",
                "dimension": dimension,
                "spearman_rho": row["spearman_rho"],
                "n": row["n"],
            }
        )

    if len(summary_expert_vs_expert):
        for _, row in summary_expert_vs_expert.iterrows():
            dimension = (
                TRIAD_RANKER_OVERALL_DIMENSION
                if row["dimension"] == "overall"
                else row["dimension"]
            )
            if dimension not in dimension_order:
                continue
            agreement_rows.append(
                {
                    "comparison": f"{row['expert_a']} vs {row['expert_b']}",
                    "dimension": dimension,
                    "spearman_rho": row["spearman_rho"],
                    "n": row["n"],
                }
            )

    combined_agreement_df = pd.DataFrame(agreement_rows)
    if combined_agreement_df.empty:
        print("No comparable agreement rows to plot.")
    else:
        expert_order = sorted(summary_expert_vs_ranker["expert"].dropna().unique())
        comparison_order = [f"{expert} vs Ranker" for expert in expert_order]
        if len(summary_expert_vs_expert):
            pair_order = (
                summary_expert_vs_expert[["expert_a", "expert_b"]]
                .drop_duplicates()
                .sort_values(["expert_a", "expert_b"])
            )
            comparison_order.extend(
                f"{row.expert_a} vs {row.expert_b}" for row in pair_order.itertuples(index=False)
            )
        comparison_order = [
            c for c in comparison_order if c in set(combined_agreement_df["comparison"])
        ]

        combined_agreement_pivot = combined_agreement_df.pivot_table(
            index="comparison", columns="dimension", values="spearman_rho", aggfunc="mean"
        ).reindex(index=comparison_order, columns=dimension_order)
        combined_agreement_n = combined_agreement_df.pivot_table(
            index="comparison", columns="dimension", values="n", aggfunc="mean"
        ).reindex(index=comparison_order, columns=dimension_order)

        display(
            combined_agreement_pivot.rename(columns=_triad_dimension_label)
            .rename_axis(index="Comparison", columns="Dimension")
            .round(3)
        )

        values = combined_agreement_pivot.to_numpy(dtype=float)
        finite_values = values[np.isfinite(values)]
        n_values = (
            pd.Series(np.ravel(combined_agreement_n.to_numpy())).dropna().astype(int).unique()
        )
        if len(n_values) == 0:
            n_note = "n unavailable"
        elif len(n_values) == 1:
            n_note = f"n={n_values[0]} for each comparison-dimension"
        else:
            n_note = f"n={n_values.min()}-{n_values.max()} across comparison-dimensions"

        if len(finite_values) == 0:
            print("Agreement values are all missing.")
        else:
            lower = -1.0 if finite_values.min() < 0 else 0.0
            has_negative = lower < 0
            norm = (
                TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)
                if has_negative
                else Normalize(vmin=0.0, vmax=1.0)
            )
            cmap = "coolwarm" if has_negative else "viridis"
            fig_width = max(5.2, 0.95 * len(dimension_order) + 2.4)
            fig_height = max(2.5, 0.48 * len(comparison_order) + 1.55)

            with plt.style.context(PAPER_STYLE), mpl.rc_context(PAPER_RC):
                fig, ax = plt.subplots(figsize=(fig_width, fig_height))
                im = ax.imshow(values, aspect="auto", cmap=cmap, norm=norm)

                for row_idx, comparison in enumerate(comparison_order):
                    for col_idx, dimension in enumerate(dimension_order):
                        value = combined_agreement_pivot.loc[comparison, dimension]
                        if not np.isfinite(value):
                            label = "NA"
                            text_color = "0.45"
                        else:
                            label = f"{value:.2f}"
                            relative = (value - lower) / (1.0 - lower) if 1.0 > lower else 1.0
                            text_color = (
                                "white"
                                if (not has_negative and relative < 0.42)
                                or (has_negative and abs(value) > 0.65)
                                else "black"
                            )
                        ax.text(
                            col_idx,
                            row_idx,
                            label,
                            ha="center",
                            va="center",
                            fontsize=7.8,
                            color=text_color,
                        )

                ax.set_xticks(np.arange(len(dimension_order)))
                ax.set_xticklabels([_triad_dimension_label(d) for d in dimension_order])
                ax.set_yticks(np.arange(len(comparison_order)))
                ax.set_yticklabels(comparison_order)
                ax.set_xlabel("Scoring dimension")
                ax.set_ylabel("Comparison")
                ax.set_title("Agreement among experts and ranker", loc="left", pad=12)
                ax.text(
                    0.0,
                    1.03,
                    f"Spearman rank correlation; {n_note}; ranker overall uses {_triad_dimension_label(TRIAD_RANKER_OVERALL_DIMENSION).replace(chr(10), ' ').lower()}.",
                    transform=ax.transAxes,
                    fontsize=7.2,
                    color="0.25",
                )
                ax.grid(False, which="both")
                ax.tick_params(length=0)
                cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
                cbar.set_label("Spearman $\\rho$")
                if sns is not None:
                    sns.despine(ax=ax, left=True, bottom=True)
                fig.tight_layout()
                plt.show()

## Outputs

- `summary_expert_vs_ranker`: one row per `(rater, dimension)`.
- `summary_expert_vs_expert`: one row per `(rater_a, rater_b, dimension)`.
- `combined_agreement_df` / `combined_agreement_pivot`: rows used by the combined rater-ranker/rater-rater agreement plot.
- `detail_by_expert`: per-paper overall rank differences for each rater.

## Notes

- Every CSV file in the ratings folder is one rater.
- With one rater, `summary_expert_vs_expert` is empty by design.
- Rater paper IDs can be either the paper key (`seed_...__paper_...`) or the full UID (`...::paper_...`).